# Runs a RAG chatbot.

The RAG chatbot will augment your queries with cat based facts pulled from tehe knowledgebase.txt vector database.

If you have not built the knoweldgebase.txt vector database, do so now by running rag_create_kb.ipynb

# Install Packages if not Present

In [1]:
!pip install ollama --quiet
!pip install import-ipynb --quiet

# Configuration values

Should I revisit and move these values into a configuration file.

In [2]:
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
KNOWLEDGE_BASE_FILE = 'knowledgebase.txt'
AUGMENT_COUNT = 5
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Logging Control.

For more detailed operation output, set 

```
logging.basicConfig(level=logging.DEBUG)
```

instead of the default logging.INFO level.

In [3]:
import logging

logging.basicConfig(level=logging.INFO)
# we suppress logging for HTTP calls
logging.getLogger('httpx').setLevel(logging.ERROR)
logging.getLogger('httpcore').setLevel(logging.ERROR)

logger = logging.getLogger(__name__)

import_ipynb permits imports from other ipynb notebooks
- Warning: the loaded notebooks capture state.  Notably the logging state.  To fix this, restart the kernel and load each imported notebook.

The UserInput retrieves queries from the console, creating a REPL like processing enviornment.

The SemanticLayer converts the facts to semantic encoded vectors.

The KnowledgeBase writes the facts to our file-based "vector database".

The RetrievalSystem fetches data from the knowledge base
- As our Knowledge Base is a file, the retreival system must implement the search for related facts
- A design choice was made to implement the Cosine distance only.  Future algorithms can be added easily.

The Augmentation component combines queries and facts to one unified set of LLM messages

The Generation component responds with the output of the Langauge Model LLM, given the query, augmented by the most relevant facts in the vector database, as determined by the semantic distance of the facts to the query.

In [12]:
import import_ipynb

import UserInput
import SemanticLayer
import KnowledgeBase
import RetrievalSystem
import Augmentation
import Generation

# UserInput queries the user for input
userInput = UserInput.UserInput(prompt='Talk to me about cats (q to quit): ', quitPhrase='q')

# SemanticLayer provides encoding services, turning text into semantic vectors.
semantic = SemanticLayer.SemanticLayer(model=EMBEDDING_MODEL)

# KnowledgeBase holds and manages encoded, semantic data
knowledge = KnowledgeBase.KnowledgeBase(filename=KNOWLEDGE_BASE_FILE)

try:    
    # RetrievalSystem fetches data from the knowledge base
    retrieval = RetrievalSystem.RetrievalSystem(knowledgeBase=knowledge)
except FileNotFoundError:
    logger.critical(f"You must run rag_create_kb.ipynb to create the knowledge base first, or use a pre-existing {KNOWLEDGE_BASE_FILE} file")
    raise

# Augmentation combines queries and facts to one unified set of LLM messages 
augmentation = Augmentation.Augmenter()

# Generation responds with the output of the Langauge Model LLM
generation = Generation.Generator(model=LANGUAGE_MODEL)

for query in userInput:
    encoding = semantic.getEncoding(text=query)
    nearestItems = retrieval.getCosineNearest(vector=encoding, count=AUGMENT_COUNT)
    augmentedMessages = augmentation.combine(query, nearestItems)
    logger.info(augmentedMessages)
    generation.respondTo(messages=augmentedMessages)


INFO:SemanticLayer:pulling model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf
INFO:SemanticLayer:  pulling manifest
INFO:SemanticLayer:  pulling 74aebb552ea7
INFO:SemanticLayer:  pulling b1899e715228
INFO:SemanticLayer:  verifying sha256 digest
INFO:SemanticLayer:  writing manifest
INFO:SemanticLayer:  success
INFO:SemanticLayer:pull of model hf.co/CompendiumLabs/bge-base-en-v1.5-gguf complete
INFO:RetrievalSystem:building cache from knowledge base
INFO:KnowledgeBase:opening knowledgebase knowledgebase.txt for reading.
CRITICAL:__main__:You must run rag_create_kb.ipynb to create the knowledge base first, or use a pre-existing knowledgebase.txt file


FileNotFoundError: [Errno 2] No such file or directory: 'knowledgebase.txt'